# Оценка раг системы 

In [1]:
import sys
from pathlib import Path

# cwd — корень репозитория, app/ лежит прямо внутри него
APP_ROOT = Path.cwd() / "app"

if not (APP_ROOT / "config.py").exists():
    raise RuntimeError(f"Не найден config.py в {APP_ROOT}. Проверь структуру проекта.")

sys.path.insert(0, str(APP_ROOT))
print("APP_ROOT:", APP_ROOT)

import pandas as pd
from services.rag_service import RAGService
from docs.rag_metrics import evaluate_retrieval

pd.set_option("display.max_colwidth", 200)

APP_ROOT: /Users/polina/vscode/beauty-routine-advisor/app


/Users/polina/vscode/beauty-routine-advisor/app/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================================
# 2. Инициализация RAG
# =========================================
rag = RAGService()
rag

Loading weights: 100%|██████████| 219/219 [00:00<00:00, 10764.71it/s]


21:28:19 [INFO] services.rag_service — ✅ RAGService: 303 векторов


In [3]:
eval_dataset = [
    {"query": "уход для сухой кожи", "skin_type": "сухая",
     "relevant_sources": {"01_dry_skin.md"}},

    {"query": "крем для сухой и стянутой кожи", "skin_type": "сухая",
     "relevant_sources": {"01_dry_skin.md"}},

    {"query": "что делать если кожа сухая после умывания", "skin_type": "сухая",
     "relevant_sources": {"01_dry_skin.md", "01_cleansing.md"}},

    {"query": "что делать при жирной коже", "skin_type": "жирная",
     "relevant_sources": {"02_oily_skin.md"}},

    {"query": "уход за жирной кожей лица", "skin_type": "жирная",
     "relevant_sources": {"02_oily_skin.md"}},

    {"query": "как уменьшить жирный блеск кожи", "skin_type": "жирная",
     "relevant_sources": {"02_oily_skin.md"}},

    {"query": "комбинированная кожа как подобрать уход", "skin_type": "комбинированная",
     "relevant_sources": {"03_combination_skin.md"}},

    {"query": "базовый ежедневный уход за лицом", "skin_type": "all",
     "relevant_sources": {"01_daily_skincare_rituals.md"}},

    {"query": "утренний и вечерний уход за кожей", "skin_type": "all",
     "relevant_sources": {"01_daily_skincare_rituals.md"}},

    {"query": "какие этапы ежедневного ухода за кожей", "skin_type": "all",
     "relevant_sources": {"01_daily_skincare_rituals.md"}},

    {"query": "как выбрать spf на каждый день", "skin_type": "all",
     "relevant_sources": {"04_spf_protection.md"}},

    {"query": "нужно ли наносить spf зимой", "skin_type": "all",
     "relevant_sources": {"04_spf_protection.md", "06_winter_skincare.md"}},

    {"query": "какой spf использовать каждый день", "skin_type": "all",
     "relevant_sources": {"04_spf_protection.md"}},

    {"query": "защита кожи от солнца каждый день", "skin_type": "all",
     "relevant_sources": {"04_spf_protection.md"}},

    {"query": "как правильно очищать кожу лица", "skin_type": "all",
     "relevant_sources": {"01_cleansing.md"}},

    {"query": "зачем нужно увлажнение кожи", "skin_type": "all",
     "relevant_sources": {"03_moisturizing.md", "01_dry_skin.md"}},

    {"query": "что должно быть в базовой рутине ухода", "skin_type": "all",
     "relevant_sources": {"01_daily_skincare_rituals.md"}},

    {"query": "как ухаживать за чувствительной кожей", "skin_type": "чувствительная",
     "relevant_sources": {"04_sensitive_skin.md"}},

    {"query": "что делать если кожа реагирует на косметику", "skin_type": "чувствительная",
     "relevant_sources": {"04_sensitive_skin.md", "03_couperose_and_rosacea.md"}},

    {"query": "как подобрать уход если кожа проблемная", "skin_type": "проблемная",
     "relevant_sources": {"01_acne_and_post_acne.md", "02_oily_skin.md"}},

    {"query": "уход при акне и постакне", "skin_type": "проблемная",
     "relevant_sources": {"01_acne_and_post_acne.md"}},

    {"query": "как избавиться от пигментации", "skin_type": "all",
     "relevant_sources": {"02_pigmentation.md"}},

    {"query": "морщины и потеря упругости кожи", "skin_type": "all",
     "relevant_sources": {"04_wrinkles_and_loss_of_firmness.md"}},

    {"query": "расширенные поры и чёрные точки", "skin_type": "all",
     "relevant_sources": {"05_blackheads_and_enlarged_pores.md"}},

    {"query": "ретинол в уходе за кожей", "skin_type": "all",
     "relevant_sources": {"01_retinol_and_derivatives.md"}},

    {"query": "витамин с для кожи польза", "skin_type": "all",
     "relevant_sources": {"02_vitamin_c.md"}},

    {"query": "ниацинамид для чего нужен", "skin_type": "all",
     "relevant_sources": {"05_niacinamide.md"}},

    {"query": "гиалуроновая кислота в косметике", "skin_type": "all",
     "relevant_sources": {"06_hyaluronic_acid.md"}},

    {"query": "aha bha кислоты для пилинга", "skin_type": "all",
     "relevant_sources": {"03_aha_bha_acids.md"}},
]

In [4]:
# =========================================
# 4. Параметры оценки
# =========================================
TOP_K = 3

In [5]:
# =========================================
# 5. Оценка по каждому запросу
# =========================================
rows = []

for item in eval_dataset:
    query = item["query"]
    skin_type = item["skin_type"]
    relevant_sources = set(item["relevant_sources"])

    chunks = rag.search(query=query, top_k=TOP_K, skin_type=skin_type)
    retrieved_sources = [chunk["source"] for chunk in chunks]
    retrieved_scores = [chunk["score"] for chunk in chunks]

    metrics = evaluate_retrieval(
        retrieved=retrieved_sources,
        relevant=relevant_sources,
        k=TOP_K,
        scores=retrieved_scores,
    )

    rows.append({
        "query": query,
        "skin_type": skin_type,
        "relevant_sources": list(relevant_sources),
        "retrieved_sources": retrieved_sources,
        "retrieved_scores": retrieved_scores,
        **metrics,
    })

results_df = pd.DataFrame(rows)
results_df

,query,skin_type,relevant_sources,retrieved_sources,retrieved_scores,precision@3,recall@3,mrr,hit_rate@3,ndcg@3,mean_score
0,уход для сухой кожи,сухая,[01_dry_skin.md],"[01_dry_skin.md, 01_dry_skin.md, 02_curly_girl_method.md]","[0.122, -0.206, -0.235]",0.6667,1.0,1.0000,1.0,1.6309,-0.1063
1,крем для сухой и стянутой кожи,сухая,[01_dry_skin.md],[01_dry_skin.md],[-0.212],0.3333,1.0,1.0000,1.0,1.0000,-0.2120
2,что делать если кожа сухая после умывания,сухая,"[01_dry_skin.md, 01_cleansing.md]","[01_dry_skin.md, 01_dry_skin.md, 01_dry_skin.md]","[-0.017, -0.07, -0.078]",1.0000,0.5,1.0000,1.0,1.3066,-0.0550
3,что делать при жирной коже,жирная,[02_oily_skin.md],"[02_oily_skin.md, 02_oily_skin.md, 02_oily_skin.md]","[0.084, 0.026, -0.048]",1.0000,1.0,1.0000,1.0,2.1309,0.0207
4,уход за жирной кожей лица,жирная,[02_oily_skin.md],"[02_oily_skin.md, 02_oily_skin.md, 02_oily_skin.md]","[0.101, 0.05, -0.151]",1.0000,1.0,1.0000,1.0,2.1309,0.0000
5,как уменьшить жирный блеск кожи,жирная,[02_oily_skin.md],"[02_oily_skin.md, 02_oily_skin.md, 02_oily_skin.md]","[-0.031, -0.067, -0.096]",1.0000,1.0,1.0000,1.0,2.1309,-0.0647
6,комбинированная кожа как подобрать уход,комбинированная,[03_combination_skin.md],"[03_combination_skin.md, 02_curly_girl_method.md, 03_combination_skin.md]","[0.122, -0.149, -0.18]",0.6667,1.0,1.0000,1.0,1.5000,-0.0690
7,базовый ежедневный уход за лицом,all,[01_daily_skincare_rituals.md],"[01_daily_skincare_rituals.md, 04_sensitive_skin.md, 01_daily_skincare_rituals.md]","[0.141, 0.093, 0.061]",0.6667,1.0,1.0000,1.0,1.5000,0.0983
8,утренний и вечерний уход за кожей,all,[01_daily_skincare_rituals.md],"[01_daily_skincare_rituals.md, 01_daily_skincare_rituals.md, 04_sensitive_skin.md]","[0.242, 0.239, 0.159]",0.6667,1.0,1.0000,1.0,1.6309,0.2133
9,какие этапы ежедневного ухода за кожей,all,[01_daily_skincare_rituals.md],"[01_daily_skincare_rituals.md, 01_cleansing.md, 01_daily_skincare_rituals.md]","[0.19, 0.077, 0.069]",0.6667,1.0,1.0000,1.0,1.5000,0.1120


In [6]:
# =========================================
# 6. Aggregate metrics
# =========================================
metric_cols = [
    f"precision@{TOP_K}",
    f"recall@{TOP_K}",
    "mrr",
    f"hit_rate@{TOP_K}",
    f"ndcg@{TOP_K}",
    "mean_score",
]

summary_df = (
    results_df[metric_cols]
    .mean()
    .round(4)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "metric"})
)

summary_df

,metric,mean_value
0,precision@3,0.5977
1,recall@3,0.8448
2,mrr,0.8563
3,hit_rate@3,0.8966
4,ndcg@3,1.3100
5,mean_score,-0.0079


In [7]:
# =========================================
# 7. Worst cases
# =========================================
worst_cases = results_df.sort_values(
    by=[f"hit_rate@{TOP_K}", "mrr", f"ndcg@{TOP_K}"],
    ascending=[True, True, True]
)

worst_cases[[
    "query",
    "skin_type",
    "relevant_sources",
    "retrieved_sources",
    f"precision@{TOP_K}",
    f"recall@{TOP_K}",
    "mrr",
    f"hit_rate@{TOP_K}",
    f"ndcg@{TOP_K}",
]]

,query,skin_type,relevant_sources,retrieved_sources,precision@3,recall@3,mrr,hit_rate@3,ndcg@3
11,нужно ли наносить spf зимой,all,"[04_spf_protection.md, 06_winter_skincare.md]","[03_combination_skin.md, 01_dry_skin.md, 04_sensitive_skin.md]",0.0000,0.0,0.0000,0.0,0.0000
19,как подобрать уход если кожа проблемная,проблемная,"[01_acne_and_post_acne.md, 02_oily_skin.md]",[],0.0000,0.0,0.0000,0.0,0.0000
20,уход при акне и постакне,проблемная,[01_acne_and_post_acne.md],[],0.0000,0.0,0.0000,0.0,0.0000
16,что должно быть в базовой рутине ухода,all,[01_daily_skincare_rituals.md],"[04_sensitive_skin.md, 03_couperose_and_rosacea.md, 01_daily_skincare_rituals.md]",0.3333,1.0,0.3333,1.0,0.5000
13,защита кожи от солнца каждый день,all,[04_spf_protection.md],"[01_daily_skincare_rituals.md, 04_spf_protection.md, 04_sensitive_skin.md]",0.3333,1.0,0.5000,1.0,0.6309
15,зачем нужно увлажнение кожи,all,"[01_dry_skin.md, 03_moisturizing.md]","[03_moisturizing.md, 01_cleansing.md, 04_spf_protection.md]",0.3333,0.5,1.0000,1.0,0.6131
1,крем для сухой и стянутой кожи,сухая,[01_dry_skin.md],[01_dry_skin.md],0.3333,1.0,1.0000,1.0,1.0000
14,как правильно очищать кожу лица,all,[01_cleansing.md],"[01_cleansing.md, 03_combination_skin.md, 01_daily_skincare_rituals.md]",0.3333,1.0,1.0000,1.0,1.0000
17,как ухаживать за чувствительной кожей,чувствительная,[04_sensitive_skin.md],[04_sensitive_skin.md],0.3333,1.0,1.0000,1.0,1.0000
18,что делать если кожа реагирует на косметику,чувствительная,"[03_couperose_and_rosacea.md, 04_sensitive_skin.md]","[04_sensitive_skin.md, 04_sensitive_skin.md]",0.6667,0.5,1.0000,1.0,1.0000


In [8]:
# =========================================
# 8. Breakdown по типу кожи
# =========================================
by_skin_type = (
    results_df.groupby("skin_type")[metric_cols]
    .mean()
    .round(4)
    .reset_index()
)

by_skin_type

,skin_type,precision@3,recall@3,mrr,hit_rate@3,ndcg@3,mean_score
0,all,0.5926,0.9167,0.8796,0.9444,1.3422,0.0154
1,жирная,1.0000,1.0000,1.0000,1.0000,2.1309,-0.0147
2,комбинированная,0.6667,1.0000,1.0000,1.0000,1.5000,-0.0690
3,проблемная,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,сухая,0.6667,0.8333,1.0000,1.0000,1.3125,-0.1244
5,чувствительная,0.5000,0.7500,1.0000,1.0000,1.0000,-0.0100


In [9]:
# =========================================
# 9. Сохранение результатов
# =========================================
RAG_PATH =  Path.cwd() / "docs"
results_df.to_csv(f"{RAG_PATH}/rag_eval_per_query.csv", index=False)
summary_df.to_csv(f"{RAG_PATH}/rag_eval_summary.csv", index=False)
by_skin_type.to_csv(f"{RAG_PATH}/rag_eval_by_skin_type.csv", index=False)

print("Saved:")
print("- rag_eval_per_query.csv")
print("- rag_eval_summary.csv")
print("- rag_eval_by_skin_type.csv")

Saved:
- rag_eval_per_query.csv
- rag_eval_summary.csv
- rag_eval_by_skin_type.csv
